# Biophysical amino-acid properties from RDKit

The biophysical modality (`datasets/biochem.py`) currently uses **placeholder** per–amino-acid tables for molecular weight, isoelectric point (pI), and Eisenberg hydrophobicity. This notebook derives the same three properties from molecular structure with **RDKit** so we can replace the placeholders with reproducible, structure-based values.

What RDKit can and can't do (checked against the RDKit docs — `rdkit.Chem.Descriptors` / `rdkit.Chem.Crippen`):

| property | RDKit? | how |
|---|---|---|
| **molecular weight** | ✅ yes | `Descriptors.MolWt(mol)` on the free amino acid, minus water → residue mass |
| **hydrophobicity** | ✅ yes | `Crippen.MolLogP(mol)` — Wildman–Crippen LogP, a computed structure-based hydrophobicity descriptor |
| **isoelectric point** | ❌ no | RDKit has **no pKa/pI predictor**; computed here from documented pKa constants via Henderson–Hasselbalch |

So MW and hydrophobicity are genuinely RDKit-computed; pI is pKa-based (RDKit only supplies the structure). Each is compared against the current placeholder before we emit a drop-in replacement.

In [1]:
import numpy as np, pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen
from datasets import biochem            # current placeholder tables, to compare against

AA = biochem.AMINO_ACIDS                # 'ACDEFGHIKLMNPQRSTVWY'
print('rdkit', rdkit.__version__)

rdkit 2026.03.4


## Amino-acid structures

Each residue is built from the SMILES of its **neutral free L-amino acid** (stereochemistry omitted — it doesn't affect MW or LogP). These are the standard 20; the MW check below is what validates that every SMILES is correct.

In [2]:
SMILES = {
    'A': 'CC(N)C(=O)O',            'R': 'NC(=N)NCCCC(N)C(=O)O',   'N': 'NC(=O)CC(N)C(=O)O',
    'D': 'OC(=O)CC(N)C(=O)O',      'C': 'OC(=O)C(N)CS',           'Q': 'NC(=O)CCC(N)C(=O)O',
    'E': 'OC(=O)CCC(N)C(=O)O',     'G': 'NCC(=O)O',               'H': 'OC(=O)C(N)Cc1cnc[nH]1',
    'I': 'CCC(C)C(N)C(=O)O',       'L': 'CC(C)CC(N)C(=O)O',       'K': 'NCCCCC(N)C(=O)O',
    'M': 'CSCCC(N)C(=O)O',         'F': 'OC(=O)C(N)Cc1ccccc1',    'P': 'OC(=O)C1CCCN1',
    'S': 'OC(=O)C(N)CO',           'T': 'CC(O)C(N)C(=O)O',        'W': 'OC(=O)C(N)Cc1c[nH]c2ccccc12',
    'Y': 'OC(=O)C(N)Cc1ccc(O)cc1', 'V': 'CC(C)C(N)C(=O)O',
}
mols = {aa: Chem.MolFromSmiles(SMILES[aa]) for aa in AA}
assert all(m is not None for m in mols.values()), 'a SMILES failed to parse'
print('built', len(mols), 'amino-acid molecules')

built 20 amino-acid molecules


## 1. Molecular weight (RDKit)

`Descriptors.MolWt` gives the average molecular weight of the free amino acid; the **residue** mass (what's added per residue in a peptide chain) is that minus one water. This should reproduce the current table almost exactly — it's the validation that the structures are right.

In [3]:
WATER = Descriptors.MolWt(Chem.MolFromSmiles('O'))   # 18.015
mw_free = {aa: Descriptors.MolWt(mols[aa]) for aa in AA}
mw_res  = {aa: mw_free[aa] - WATER for aa in AA}
df_mw = pd.DataFrame({'free_MW': mw_free, 'residue_MW': mw_res, 'current': biochem._MW})
df_mw['diff'] = df_mw['residue_MW'] - df_mw['current']
print(f'max |residue_MW - current| = {df_mw["diff"].abs().max():.4f} Da')
df_mw.round(3)

max |residue_MW - current| = 0.0074 Da


,free_MW,residue_MW,current,diff
A,89.094,71.079,71.079,0.000
C,121.161,103.146,103.139,0.007
D,133.103,115.088,115.089,-0.001
E,147.130,129.115,129.116,-0.000
F,165.192,147.177,147.177,0.000
G,75.067,57.052,57.052,0.000
H,155.157,137.142,137.141,0.001
I,131.175,113.160,113.159,0.001
K,146.190,128.175,128.174,0.001
L,131.175,113.160,113.159,0.001


## 2. Hydrophobicity (RDKit Crippen LogP)

`Crippen.MolLogP` is the Wildman–Crippen (1999) octanol–water LogP — a computed, structure-based hydrophobicity. It is a **different scale** from Eisenberg's experimental consensus values (different units, computed vs. measured), so we don't expect the numbers to match — we expect them to *correlate*. Both rank hydrophobic residues (I, L, F, W, V) high and charged/polar ones (R, D, E, K, N) low.

In [4]:
logp = {aa: Crippen.MolLogP(mols[aa]) for aa in AA}
df_h = pd.DataFrame({'rdkit_logP': logp, 'eisenberg_current': biochem._HYDRO_EISENBERG})
r = np.corrcoef([logp[a] for a in AA], [biochem._HYDRO_EISENBERG[a] for a in AA])[0, 1]
print(f'Pearson r (RDKit LogP vs Eisenberg) = {r:.3f}')
df_h.sort_values('rdkit_logP', ascending=False).round(3)

Pearson r (RDKit LogP vs Eisenberg) = 0.691


,rdkit_logP,eisenberg_current
W,1.122,0.81
F,0.641,1.19
I,0.444,1.38
L,0.444,1.06
Y,0.347,0.26
M,0.151,0.64
V,0.054,1.08
P,-0.177,0.12
K,-0.473,-1.50
A,-0.582,0.62


## 3. Isoelectric point (pKa-based — not an RDKit descriptor)

RDKit does not predict pKa, so pI can't come from RDKit directly. We compute it the standard way: net charge as a function of pH (Henderson–Hasselbalch) over the terminal groups plus any ionizable side chain, solved for the pH where net charge is zero. pKa constants are documented literature values (Lehninger / EMBOSS-style).

In [5]:
CTERM, NTERM = 2.34, 9.60          # generic alpha-carboxyl / alpha-amino pKa
POS_SIDE = {'K': 10.53, 'R': 12.48, 'H': 6.00}
NEG_SIDE = {'D': 3.65, 'E': 4.25, 'C': 8.33, 'Y': 10.07}

def net_charge(aa, pH):
    pos = [NTERM] + ([POS_SIDE[aa]] if aa in POS_SIDE else [])
    neg = [CTERM] + ([NEG_SIDE[aa]] if aa in NEG_SIDE else [])
    return (sum(1 / (1 + 10 ** (pH - pk)) for pk in pos)
            - sum(1 / (1 + 10 ** (pk - pH)) for pk in neg))

def isoelectric_point(aa):
    lo, hi = 0.0, 14.0                 # bisection on a monotonic net-charge curve
    for _ in range(100):
        mid = (lo + hi) / 2
        if net_charge(aa, mid) > 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

pi = {aa: isoelectric_point(aa) for aa in AA}
df_pi = pd.DataFrame({'pI_computed': pi, 'current': biochem._PI})
df_pi['diff'] = df_pi['pI_computed'] - df_pi['current']
print(f'max |pI_computed - current| = {df_pi["diff"].abs().max():.2f}')
df_pi.round(2)

max |pI_computed - current| = 0.56


,pI_computed,current,diff
A,5.97,6.00,-0.03
C,5.32,5.07,0.25
D,2.99,2.77,0.22
E,3.29,3.22,0.07
F,5.97,5.48,0.49
G,5.97,5.97,-0.00
H,7.80,7.59,0.21
I,5.97,6.02,-0.05
K,10.07,9.74,0.33
L,5.97,5.98,-0.01


## 4. Assemble & z-score

Same layout the model consumes: three per-residue channels, each z-scored over the 20 canonical amino acids (identical to `biochem._zscore`).

In [6]:
def zscore(d):
    v = np.array([d[a] for a in AA])
    return {a: float((d[a] - v.mean()) / v.std()) for a in AA}

MW_Z, PI_Z, HYDRO_Z = zscore(mw_res), zscore(pi), zscore(logp)
final = pd.DataFrame({
    'residue_MW': mw_res, 'pI': pi, 'logP': logp,
    'MW_z': MW_Z, 'pI_z': PI_Z, 'hydro_z': HYDRO_Z,
})
final.round(3)

,residue_MW,pI,logP,MW_z,pI_z,hydro_z
A,71.079,5.970,-0.582,-1.589,-0.133,-0.141
C,103.146,5.324,-0.672,-0.523,-0.502,-0.254
D,115.088,2.995,-1.127,-0.126,-1.834,-0.827
E,129.115,3.295,-0.737,0.340,-1.662,-0.336
F,147.177,5.970,0.641,0.940,-0.133,1.398
G,57.052,5.970,-0.970,-2.055,-0.133,-0.629
H,137.142,7.800,-0.636,0.607,0.914,-0.209
I,113.160,5.970,0.444,-0.190,-0.133,1.150
K,128.175,10.065,-0.473,0.309,2.209,-0.003
L,113.160,5.970,0.444,-0.190,-0.133,1.150


## 5. Drop-in replacement for `datasets/biochem.py`

The cell below prints the three raw dicts formatted exactly like the ones in `datasets/biochem.py` (`_MW`, `_PI`, `_HYDRO_EISENBERG`). Paste them in to swap the placeholders for the RDKit-derived values. **Note the hydrophobicity dict is now Crippen LogP, not Eisenberg** — that's a deliberate methodological change (a computed scale in place of the experimental one); keep the variable name or rename it as you prefer. MW and pI are drop-in equivalents to what's already there.

In [7]:
def fmt(name, d, prec):
    items = [f'"{a}": {round(d[a], prec)}' for a in AA]
    lines, cur = [], '    '
    for it in items:
        piece = it + ', '
        if len(cur) + len(piece) > 78:
            lines.append(cur.rstrip()); cur = '    '
        cur += piece
    lines.append(cur.rstrip().rstrip(','))
    return f'{name} = {{\n' + '\n'.join(lines) + '\n}'

print('# --- RDKit-derived amino-acid properties (paste into datasets/biochem.py) ---\n')
print(fmt('_MW', mw_res, 4), '\n')
print(fmt('_PI', pi, 2), '\n')
print(fmt('_HYDRO_CRIPPEN_LOGP', logp, 3))

# --- RDKit-derived amino-acid properties (paste into datasets/biochem.py) ---

_MW = {
    "A": 71.079, "C": 103.146, "D": 115.088, "E": 129.115, "F": 147.177,
    "G": 57.052, "H": 137.142, "I": 113.16, "K": 128.175, "L": 113.16,
    "M": 131.2, "N": 114.104, "P": 97.117, "Q": 128.131, "R": 156.189,
    "S": 87.078, "T": 101.105, "V": 99.133, "W": 186.214, "Y": 163.176
} 

_PI = {
    "A": 5.97, "C": 5.32, "D": 2.99, "E": 3.29, "F": 5.97, "G": 5.97,
    "H": 7.8, "I": 5.97, "K": 10.07, "L": 5.97, "M": 5.97, "N": 5.97,
    "P": 5.97, "Q": 5.97, "R": 11.04, "S": 5.97, "T": 5.97, "V": 5.97,
    "W": 5.97, "Y": 5.91
} 

_HYDRO_CRIPPEN_LOGP = {
    "A": -0.582, "C": -0.672, "D": -1.127, "E": -0.737, "F": 0.641,
    "G": -0.97, "H": -0.636, "I": 0.444, "K": -0.473, "L": 0.444, "M": 0.151,
    "N": -1.726, "P": -0.177, "Q": -1.336, "R": -1.338, "S": -1.609,
    "T": -1.221, "V": 0.054, "W": 1.122, "Y": 0.347
}


### How to wire it in

1. Replace `_MW` and `_PI` in `datasets/biochem.py` with the printed dicts — these are equivalent (MW validated to <0.01 Da; pI within the pKa-set tolerance).
2. For hydrophobicity, decide: keep Eisenberg, or switch to the RDKit `_HYDRO_CRIPPEN_LOGP` above. LogP is fully reproducible from structure; Eisenberg is the scale the Kulkarni et al. paper names. If you switch, rename `_HYDRO_EISENBERG` → `_HYDRO` and update `_HYDRO_Z`.
3. Nothing downstream changes — `AA_PROPERTY`, `biophysical_matrix`, and the whole `biophysical` modality read from these three dicts.

**Caveats**
- LogP here is the *free amino acid*; a more in-chain-faithful hydrophobicity would use a capped residue (N-acetyl … methylamide). Easy refinement if wanted.
- pI is pKa-set-dependent; swapping the constants (EMBOSS vs. Lehninger vs. Bjellqvist) shifts values by a few tenths — worth matching whatever set the reference uses.
- Still worth confirming Kulkarni et al.'s exact table/normalization (see `TODO.md`); this gives a principled, reproducible default in the meantime.